# 03 - Test API
Sends requests to the running server. Start the server in `02_API_Server` first (Step 6) and keep it open.

In [ ]:
import requests

BASE_URL = "http://127.0.0.1:8000"

## Health check

In [ ]:
# Is the server up?
try:
    r = requests.get(f"{BASE_URL}/health", timeout=5)
    print(r.status_code, r.json())
except requests.exceptions.ConnectionError:
    print("Cannot connect. Run Step 6 in 02_API_Server.ipynb and keep it open.")

## Model info

In [ ]:
# Which model is being used?
requests.get(f"{BASE_URL}/model-info").json()

## Predict for one person
Change the values and run again.

In [ ]:
# Person's details
person = {
    "Has_Patches": 1,
    "Has_MS": 0,
    "Owner_Annual_Income_Lakhs": 4.5,
}

# Send to the API and read the answer
result = requests.post(f"{BASE_URL}/predict", json=person).json()

print("Input:", person)
print("Needs loan:", result["requires_loan"])
print("Probability:", result["probability"])
print("Risk band:", result["risk_band"])

## Predict for many people

In [ ]:
# Several people in one request
people = {"applications": [
    {"Has_Patches": 0, "Has_MS": 1, "Owner_Annual_Income_Lakhs": 15},
    {"Has_Patches": 1, "Has_MS": 1, "Owner_Annual_Income_Lakhs": 3},
    {"Has_Patches": 1, "Has_MS": 0, "Owner_Annual_Income_Lakhs": 5},
]}

response = requests.post(f"{BASE_URL}/predict/batch", json=people).json()

# Print each person with their result
for p, pred in zip(people["applications"], response["predictions"]):
    print(p, "->", pred["requires_loan"], pred["risk_band"])

## Invalid input
A `422` status means the API correctly rejected bad data.

In [ ]:
# Invalid values: Has_Patches must be 0/1 and income can't be negative
bad = {"Has_Patches": 5, "Has_MS": 0, "Owner_Annual_Income_Lakhs": -1}
r = requests.post(f"{BASE_URL}/predict", json=bad)
print("Status code:", r.status_code)